## S&P 100 Portfolio Optimization Project

**PROJECT OVERVIEW**: This notebook implements an in-depth portfolio analysis on S&P 100 constituents (2015-2024).

METHODOLOGIES
- Risk-adjusted performance metrics using dynamic Treasury rates
- Sector-level analysis with equal and market-cap weighting
- Rolling statistics and regime analysis
- Pairs trading identification and correlation analysis
- Distribution analysis including skewness, kurtosis, and maximum drawdown

KEY RESULTS
- Analyzed 100 stocks across 11 sectors over 2,138 trading days
- Identified 27 stocks with Sharpe ratios > 0.7
- Technology sector outperformance: 27.2% annual return, 0.81 Sharpe
- Average portfolio Sharpe: 0.54 (risk-adjusted with dynamic Treasury rates)
- Sector correlation analysis reveals diversification opportunities (avg correlation: 0.60)

# Portfolio Data Exploration and Analysis

This notebook demonstrates how to load, explore, and analyze financial data for portfolio optimization.

## 0. Importing Libraries

In [ ]:
# Import required libraries
import sys
import os
sys.path.append('..')  # Add parent directory to path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import requests

import yfinance as yf
import json

# Import our modules
from src.data.fetcher import DataFetcher

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load Historical Data

In [ ]:
#Using Wikiperida to get the tickers of S&P100

url = 'https://en.wikipedia.org/wiki/S%26P_100'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
response = requests.get(url, headers=headers)
tables = pd.read_html(response.text)
sp100_tickers = tables[2]['Symbol'].tolist()  # Check which table index has the tickers

In [ ]:
#Correcting the ticker for BRK
#Removing PLTR because it went public on Sep 30, 2020 while the rest of the stocks have data going back to 2015
sp100_tickers = ['BRK-B' if t == 'BRK.B' else t for t in sp100_tickers if t != 'PLTR']

In [ ]:
#Trial case for simplicity
#tickers = ['AAPL', 'GOOG', 'MSFT', 'AMZN', 'JPM']

tickers = sp100_tickers
start_date = '2015-01-01'
end_date = '2024-01-01'

# Fetch data
fetcher = DataFetcher()
prices = fetcher.fetch_price_data(tickers, start_date, end_date)

print(f"Data shape: {prices.shape}")
print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
print(f"\nFirst few rows:")
prices.head()

## 2. Calculate Returns

### 2.1 Computing Risk-Free Rates Using Treasury Data

In [ ]:
# Fetch Treasury data
tbill_data = yf.download('^IRX', start=start_date, end=end_date, progress=False)

# Access Close column from multi-index dataframe
tbill_rates = tbill_data[('Close', '^IRX')] / 100  # Convert percentage to decimal

# Calculate average risk-free rate
RISK_FREE_RATE = tbill_rates.mean()

print(f"Successfully fetched {len(tbill_data)} days of Treasury data")
print(f"\nRate Statistics (2015-2024):")
print(f"Average rate: {RISK_FREE_RATE:.3%}")
print(f"Starting rate (Jan 2015): {tbill_rates.iloc[0]:.3%}")
print(f"Ending rate (Dec 2023): {tbill_rates.iloc[-1]:.3%}")
print(f"Minimum rate: {tbill_rates.min():.3%} (near-zero policy)")
print(f"Maximum rate: {tbill_rates.max():.3%} (recent hikes)")

# Show rate evolution
print(f"\nRate Environment Changes:")
print(f"2015-2017 avg: {tbill_rates['2015':'2019'].mean():.3%}")
print(f"2020-2021 avg: {tbill_rates['2020':'2021'].mean():.3%} (COVID)")
print(f"2022-2023 avg: {tbill_rates['2022':'2023'].mean():.3%} (Fed hikes)")

### 2.2 Computing Returns and Excess Returns

In [ ]:
## Calculate Returns and Prepare Monthly Risk-Free Rates

# Calculate daily returns
returns = prices.pct_change().dropna()

# Create monthly groupings
returns_monthly = returns.resample('M').apply(lambda x: (1 + x).prod() - 1)

# Convert annual risk-free rates to monthly
tbill_monthly_annual = tbill_rates.resample('M').mean()  # Still annual rates
tbill_monthly = tbill_monthly_annual / 12  # Convert to monthly

# Ensure alignment
aligned_dates = returns_monthly.index.intersection(tbill_monthly.index)
returns_monthly = returns_monthly.loc[aligned_dates]
tbill_monthly = tbill_monthly.loc[aligned_dates]

print("Monthly Data Preparation Complete")
print("="*50)
print(f"Total months analyzed: {len(returns_monthly)}")

# Show risk-free rates (now monthly)
print(f"\nMonthly Risk-Free Rate Statistics:")
print(f"Average (monthly): {tbill_monthly.mean():.4%}")
print(f"Average (annualized): {tbill_monthly.mean()*12:.3%}")

# Calculate monthly excess returns with MONTHLY risk-free rate
excess_returns_monthly = returns_monthly.subtract(tbill_monthly, axis=0)

print(f"\nExcess Returns Calculated:")
print(f"Average monthly excess return: {excess_returns_monthly.mean().mean():.3%}")

### 2.3 Sharpe Ratios for Stocks

In [ ]:
## Calculate Sharpe Ratios with Dynamic Risk-Free Rates

# Annualize monthly excess returns and volatility
annual_excess_returns = excess_returns_monthly.mean() * 12
annual_volatility = excess_returns_monthly.std() * np.sqrt(12)

# Calculate Sharpe ratios using dynamic risk-free rates
sharpe_ratios_dynamic = annual_excess_returns / annual_volatility

# For comparison, calculate traditional Sharpe with static rate
annual_returns = returns_monthly.mean() * 12
sharpe_ratios_static = (annual_returns - RISK_FREE_RATE) / annual_volatility

# Create comprehensive summary
summary = pd.DataFrame({
    'Annual Return': annual_returns,
    'Annual Volatility': annual_volatility,
    'Excess Return (Dynamic RF)': annual_excess_returns,
    'Sharpe (Dynamic RF)': sharpe_ratios_dynamic,
    'Sharpe (Static RF)': sharpe_ratios_static,
    'Sharpe Difference': sharpe_ratios_dynamic - sharpe_ratios_static
})

summary = summary.sort_values('Sharpe (Dynamic RF)', ascending=False)

print("Sharpe Ratio Comparison")
print("="*50)
print(f"Using Dynamic RF - Average Sharpe: {sharpe_ratios_dynamic.mean():.3f}")
print(f"Using Static RF - Average Sharpe: {sharpe_ratios_static.mean():.3f}")
print(f"Average difference: {(sharpe_ratios_dynamic - sharpe_ratios_static).mean():.3f}")

print(f"\nTop 5 Stocks by Sharpe (Dynamic RF):")
for ticker in summary.head(5).index:
    print(f"  {ticker}: {summary.loc[ticker, 'Sharpe (Dynamic RF)']:.3f}")

print(f"\nLargest Sharpe Improvements from Dynamic RF:")
top_improvements = summary.nlargest(3, 'Sharpe Difference')
for ticker in top_improvements.index:
    print(f"  {ticker}: +{top_improvements.loc[ticker, 'Sharpe Difference']:.3f}")

### 2.4 Daily Returns Distribution Analysis

In [ ]:
## Distribution and Risk Characteristics

from scipy import stats

# Calculate distribution metrics
skewness = returns.skew()
kurtosis = returns.kurtosis()

# Downside deviation (for Sortino ratio)
downside_returns = returns[returns < 0]
downside_deviation = downside_returns.std() * np.sqrt(252)
sortino_ratios = (annual_returns - RISK_FREE_RATE) / downside_deviation

# Maximum drawdown
cumulative_returns = (1 + returns).cumprod()
running_max = cumulative_returns.expanding().max()
drawdowns = (cumulative_returns - running_max) / running_max
max_drawdowns = drawdowns.min()

# Add to summary
summary['Skewness'] = skewness
summary['Kurtosis'] = kurtosis
summary['Sortino'] = sortino_ratios
summary['Max Drawdown'] = max_drawdowns

print("Risk Distribution Analysis")
print("="*50)
print(f"Average Skewness: {skewness.mean():.3f} (negative = left tail)")
print(f"Average Kurtosis: {kurtosis.mean():.3f} (>0 = fat tails)")
print(f"Stocks with negative skew: {(skewness < 0).sum()}/{len(skewness)}")
print(f"Stocks with excess kurtosis: {(kurtosis > 3).sum()}/{len(kurtosis)}")

print(f"\nDownside Risk Metrics:")
print(f"Average Sortino Ratio: {sortino_ratios.mean():.3f}")
print(f"Average Max Drawdown: {max_drawdowns.mean():.1%}")
print(f"Worst drawdown: {max_drawdowns.min():.1%} ({max_drawdowns.idxmin()})")

## 3. Visualize Price Trends

### 3.1 Top 5 Performers

In [ ]:
#Visualizing best and worst performers

# Normalize prices to start at 100
normalized_prices = prices / prices.iloc[0] * 100

# Select top and bottom performers
top_5 = annual_returns.nlargest(5).index
bottom_5 = annual_returns.nsmallest(5).index

In [ ]:

# Create plot for top performers
fig = go.Figure()

for ticker in top_5:
    fig.add_trace(go.Scatter(
        x=normalized_prices.index,
        y=normalized_prices[ticker],
        mode='lines',
        name=f'{ticker} ({annual_returns[ticker]:.1%})',
        line=dict(width=2)
    ))

# Add S&P 100 average
fig.add_trace(go.Scatter(
    x=normalized_prices.index,
    y=normalized_prices.mean(axis=1),
    mode='lines',
    name='S&P 100 Avg',
    line=dict(width=2, color='black', dash='dash')
))

fig.update_layout(
    title=f'Top 5 Performers (Avg Return: {annual_returns[top_5].mean():.1%})',
    xaxis_title='Date',
    yaxis_title='Normalized Price (Base=100)',
    hovermode='x unified',
    template='plotly_white',
    height=400
)
fig.show()

print("Top 5 Performers:")
for ticker in top_5:
    print(f"  {ticker:6s}: {annual_returns[ticker]:6.1%} return")

### 3.2 Bottom 5 Performers

In [ ]:
## Visualize Bottom 5 Performers

# Create plot for bottom performers
fig = go.Figure()

for ticker in bottom_5:
    fig.add_trace(go.Scatter(
        x=normalized_prices.index,
        y=normalized_prices[ticker],
        mode='lines',
        name=f'{ticker} ({annual_returns[ticker]:.1%})',
        line=dict(width=2)
    ))

# Add S&P 100 average
fig.add_trace(go.Scatter(
    x=normalized_prices.index,
    y=normalized_prices.mean(axis=1),
    mode='lines',
    name='S&P 100 Avg',
    line=dict(width=2, color='black', dash='dash')
))

fig.update_layout(
    title=f'Bottom 5 Performers (Avg Return: {annual_returns[bottom_5].mean():.1%})',
    xaxis_title='Date',
    yaxis_title='Normalized Price (Base=100)',
    hovermode='x unified',
    template='plotly_white',
    height=400
)
fig.show()

print("\nBottom 5 Performers:")
for ticker in bottom_5:
    print(f"  {ticker:6s}: {annual_returns[ticker]:6.1%} return")
    
print(f"\nS&P 100 Average Return: {annual_returns.mean():.1%}")

### 3.3 Volatility Trends (Most and Least Volatile Stocks)

In [ ]:
## Volatility Analysis - High vs Low Volatility Stocks

# Identify high and low volatility stocks
high_vol_stocks = annual_volatility.nlargest(5).index
low_vol_stocks = annual_volatility.nsmallest(5).index

# Calculate rolling volatility (60-day window)
rolling_vol = returns.rolling(60).std() * np.sqrt(252)

In [ ]:
# Create subplots
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        f'High Volatility Stocks (Avg: {annual_volatility[high_vol_stocks].mean():.1%})',
        f'Low Volatility Stocks (Avg: {annual_volatility[low_vol_stocks].mean():.1%})'
    ),
    vertical_spacing=0.12
)

# Plot high volatility stocks
for i, ticker in enumerate(high_vol_stocks):
    fig.add_trace(
        go.Scatter(x=rolling_vol.index, y=rolling_vol[ticker],
                   name=f'{ticker} ({annual_volatility[ticker]:.1%})',
                   mode='lines', line=dict(width=1.5),
                   legendgroup='high',
                   legendgrouptitle_text="High Volatility"),
        row=1, col=1
    )

# Plot low volatility stocks  
for i, ticker in enumerate(low_vol_stocks):
    fig.add_trace(
        go.Scatter(x=rolling_vol.index, y=rolling_vol[ticker],
                   name=f'{ticker} ({annual_volatility[ticker]:.1%})',
                   mode='lines', line=dict(width=1.5),
                   legendgroup='low',
                   legendgrouptitle_text="Low Volatility"),
        row=2, col=1
    )

fig.update_yaxes(title_text="Annualized Volatility", row=1, col=1)
fig.update_yaxes(title_text="Annualized Volatility", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.update_layout(
    height=700,
    title_text="Rolling 60-Day Volatility: High vs Low Volatility Stocks",
    hovermode='x unified',
    legend=dict(
        groupclick="toggleitem",
        tracegroupgap=180
    )
)
fig.show()

In [ ]:
# Summary statistics
print("Volatility Analysis Summary")
print("-"*50)
print("High Volatility Stocks:")
for ticker in high_vol_stocks:
    print(f"  {ticker}: {annual_volatility[ticker]:.1%} (Return: {annual_returns[ticker]:.1%})")

print("\nLow Volatility Stocks:")
for ticker in low_vol_stocks:
    print(f"  {ticker}: {annual_volatility[ticker]:.1%} (Return: {annual_returns[ticker]:.1%})")

## 4. Sector-wise Performance

### 4.1 Sector-Based Split

In [ ]:
# Function to fetch and cache sector information
def fetch_sector_mappings(tickers, cache_file='../data/cache/sector_mappings.json'):
    """
    Fetch sector information for all tickers with caching
    """
    sector_mapping = {}
    
    # Try to load from cache first
    if os.path.exists(cache_file):
        try:
            with open(cache_file, 'r') as f:
                cached_data = json.load(f)
                print(f"✓ Loaded cached sector data for {len(cached_data)} tickers")
                sector_mapping = cached_data
        except:
            pass
    
    # Fetch missing sectors
    missing_tickers = [t for t in tickers if t not in sector_mapping]
    
    if missing_tickers:
        print(f"\n Fetching sector data for {len(missing_tickers)} tickers...")
        for i, ticker in enumerate(missing_tickers):
            try:
                info = yf.Ticker(ticker).info
                sector = info.get('sector', 'Unknown')
                sector_mapping[ticker] = sector
                
                if (i + 1) % 10 == 0:
                    print(f"   Processed {i + 1}/{len(missing_tickers)} tickers...")
                    
            except Exception as e:
                print(f"    Could not fetch sector for {ticker}")
                sector_mapping[ticker] = 'Unknown'
        
        # Save to cache
        try:
            os.makedirs(os.path.dirname(cache_file), exist_ok=True)
            with open(cache_file, 'w') as f:
                json.dump(sector_mapping, f, indent=2)
            print(f"✓ Saved sector mappings to cache")
        except:
            pass
    
    return sector_mapping

In [ ]:
# Fetch sector mappings for all tickers
sector_mapping = fetch_sector_mappings(tickers)

# Create sector-based groupings
sectors = {}
for ticker in tickers:
    sector = sector_mapping.get(ticker, 'Unknown')
    if sector not in sectors:
        sectors[sector] = []
    sectors[sector].append(ticker)

# Remove 'Unknown' sector if it exists and is small
if 'Unknown' in sectors and len(sectors['Unknown']) < 3:
    unknown_tickers = sectors.pop('Unknown')
    print(f"\n Removed {len(unknown_tickers)} tickers with unknown sectors: {unknown_tickers}")

# Sort sectors by number of stocks
sectors = dict(sorted(sectors.items(), key=lambda x: len(x[1]), reverse=True))

In [ ]:
# Display sector distribution
print(f"\n Sector Distribution Summary:")
print("="*60)
total_stocks = sum(len(stocks) for stocks in sectors.values())
print(f"Total Stocks: {total_stocks}")
print(f"Number of Sectors: {len(sectors)}\n")

print(f"{'Sector':<30} {'Count':>8} {'Percentage':>12} {'Tickers'}")
print("-"*75)

for sector, stocks in sectors.items():
    pct = len(stocks) / total_stocks * 100
    ticker_list = ', '.join(stocks[:5])  # Show first 5 tickers
    if len(stocks) > 5:
        ticker_list += f", ... (+{len(stocks)-5} more)"
    print(f"{sector:<30} {len(stocks):>8} {pct:>11.1f}% {ticker_list}")

# Create DataFrame for easier manipulation
sector_df = pd.DataFrame([
    {'Ticker': ticker, 'Sector': sector}
    for sector, stocks in sectors.items()
    for ticker in stocks
])

# Merge with returns data
sector_summary = pd.merge(
    sector_df,
    summary,  # This should have Annual Return, Annual Volatility, Sharpe Ratio
    left_on='Ticker',
    right_index=True
)

print(f"\n Sector classification complete!")
print(f"   • Successfully classified {len(sector_summary)} stocks")
print(f"   • Ready for performance analysis")

# Store sectors dictionary for use in next cells
print(f"\n Variables saved for next analysis:")
print(f"   • sectors: Dictionary with sector -> [tickers] mapping")
print(f"   • sector_mapping: Dictionary with ticker -> sector mapping")
print(f"   • sector_summary: DataFrame with all ticker metrics and sectors")

### 4.2 Sector Performance - Equal Weighted

In [ ]:
# Check what columns are available
print("Available columns in summary:", summary.columns.tolist())
print("Available columns in sector_summary:", sector_summary.columns.tolist())

In [ ]:
## Calculate Equal-Weighted Sector Performance

# Calculate sector-level metrics using equal weighting
sector_metrics = {}

for sector, ticker_list in sectors.items():
    # Get data for stocks in this sector
    sector_data = sector_summary[sector_summary['Sector'] == sector]
    
    if len(sector_data) > 0:
        # Equal-weighted averages - use the correct column name
        sector_metrics[sector] = {
            'Return': sector_data['Annual Return'].mean(),
            'Volatility': sector_data['Annual Volatility'].mean(),
            'Sharpe': sector_data['Sharpe (Static RF)'].mean(),  # Using Static RF for consistency
            'Count': len(sector_data),
            'Best Stock': sector_data.nlargest(1, 'Sharpe (Static RF)').index[0],
            'Worst Stock': sector_data.nsmallest(1, 'Sharpe (Static RF)').index[0]
        }

# Convert to DataFrame and sort by Sharpe
sector_performance = pd.DataFrame(sector_metrics).T
sector_performance = sector_performance.sort_values('Sharpe', ascending=False)

# Display results
print("Equal-Weighted Sector Performance")
print("="*80)
print(f"{'Sector':<25} {'Return':>10} {'Volatility':>12} {'Sharpe':>10} {'Stocks':>8}")
print("-"*80)

for sector, metrics in sector_performance.iterrows():
    print(f"{sector:<25} {metrics['Return']:>9.1%} {metrics['Volatility']:>11.1%} "
          f"{metrics['Sharpe']:>10.3f} {metrics['Count']:>8.0f}")

# Summary statistics
print(f"\nKey Insights:")
print(f"Best performing sector: {sector_performance.index[0]} (Sharpe: {sector_performance.iloc[0]['Sharpe']:.3f})")
print(f"Worst performing sector: {sector_performance.index[-1]} (Sharpe: {sector_performance.iloc[-1]['Sharpe']:.3f})")
print(f"Highest return sector: {sector_performance['Return'].idxmax()} ({sector_performance['Return'].max():.1%})")
print(f"Lowest risk sector: {sector_performance['Volatility'].idxmin()} ({sector_performance['Volatility'].min():.1%})")

### 4.3 Correlation Heatmap

In [ ]:
## Sector Correlation Analysis

# Create sector-level returns (equal-weighted)
sector_returns = pd.DataFrame()
for sector, ticker_list in sectors.items():
    sector_stocks = [t for t in ticker_list if t in returns.columns]
    if len(sector_stocks) > 0:
        sector_returns[sector] = returns[sector_stocks].mean(axis=1)

# Calculate correlation matrix
sector_correlation = sector_returns.corr()

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=sector_correlation.values,
    x=[s[:20] for s in sector_correlation.columns],  # Truncate long names
    y=[s[:20] for s in sector_correlation.columns],
    text=np.round(sector_correlation.values, 2),
    texttemplate='%{text}',
    textfont={"size": 10},
    colorscale='RdBu',
    zmid=0.5,
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title="Sector Correlation Matrix (Equal-Weighted)",
    height=600,
    width=700,
    xaxis={'side': 'bottom'},
    yaxis={'side': 'left'}
)
fig.show()

# Key insights
corr_values = sector_correlation.values[np.triu_indices_from(sector_correlation.values, k=1)]
print(f"Correlation Statistics:")
print(f"Average sector correlation: {np.mean(corr_values):.3f}")
print(f"Std dev of correlations: {np.std(corr_values):.3f}")

# Find most diversifying sectors
avg_correlations = sector_correlation.mean() - 1/len(sector_correlation)  # Exclude self
print(f"\nMost diversifying sectors (lowest avg correlation):")
for sector in avg_correlations.nsmallest(3).index:
    print(f"  {sector}: {avg_correlations[sector]:.3f}")

### 4.4 Risk-Return Scatter

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Individual Stocks (Colored by Sector)', 'Sector Aggregates'),
    horizontal_spacing=0.12
)

# 1. Individual stocks scatter
for sector in sectors.keys():
    sector_stocks = sector_summary[sector_summary['Sector'] == sector]
    fig.add_trace(
        go.Scatter(
            x=sector_stocks['Annual Volatility'],
            y=sector_stocks['Annual Return'],
            mode='markers',
            name=sector[:20],  # Truncate long names
            marker=dict(size=8, opacity=0.7),
            text=sector_stocks['Ticker'],
            hovertemplate='%{text}<br>Return: %{y:.1%}<br>Vol: %{x:.1%}'
        ),
        row=1, col=1
    )

# 2. Sector-level scatter
fig.add_trace(
    go.Scatter(
        x=sector_performance['Volatility'],
        y=sector_performance['Return'],
        mode='markers+text',
        text=[s[:10] for s in sector_performance.index],
        textposition='top center',
        marker=dict(
            size=sector_performance['Count']*2,  # Size by number of stocks
            color=sector_performance['Sharpe'],
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(title="Sharpe", x=1.15)
        ),
        showlegend=False,
        hovertemplate='%{text}<br>Return: %{y:.1%}<br>Vol: %{x:.1%}<br>Stocks: %{marker.size}'
    ),
    row=1, col=2
)

# Add efficient frontier reference line (approximation)
fig.add_shape(type="line", row=1, col=1,
    x0=0.15, y0=0.05, x1=0.35, y1=0.30,
    line=dict(color="gray", width=1, dash="dash")
)

fig.update_xaxes(title_text="Volatility", row=1, col=1)
fig.update_xaxes(title_text="Volatility", row=1, col=2)
fig.update_yaxes(title_text="Annual Return", row=1, col=1)
fig.update_yaxes(title_text="Annual Return", row=1, col=2)

fig.update_layout(height=500, width=1200, title_text="Risk-Return Analysis")
fig.show()

In [ ]:
# Identify outliers
print("Notable Positions:")
print("-"*50)

# Best risk-adjusted (high Sharpe)
best_sharpe = sector_summary.nlargest(3, 'Sharpe (Static RF)')
print("Best Risk-Adjusted Stocks:")
for ticker in best_sharpe['Ticker']:
    print(f"  {ticker}: Return={annual_returns[ticker]:.1%}, Vol={annual_volatility[ticker]:.1%}")

# High return, high risk
high_return = sector_summary[sector_summary['Annual Return'] > 0.25]
if len(high_return) > 0:
    print(f"\nHigh Return Stocks (>25%):")
    for _, row in high_return.iterrows():
        print(f"  {row['Ticker']}: Return={row['Annual Return']:.1%}, Vol={row['Annual Volatility']:.1%}")

## 5. Rolling Statistics

### 5.1 For Stocks

In [ ]:
# Calculate rolling statistics
window = 252  # 1 year

# Select one ticker for detailed analysis
ticker = 'AAPL'

rolling_mean = returns[ticker].rolling(window).mean() * 252
rolling_std = returns[ticker].rolling(window).std() * np.sqrt(252)
rolling_sharpe = rolling_mean / rolling_std

# Create subplots
fig = make_subplots(rows=3, cols=1, 
                    subplot_titles=('Rolling Annual Return', 
                                   'Rolling Annual Volatility', 
                                   'Rolling Sharpe Ratio'),
                    shared_xaxes=True)

# Rolling return
fig.add_trace(go.Scatter(x=rolling_mean.index, y=rolling_mean, 
                         name='Annual Return', line=dict(color='blue')), 
              row=1, col=1)

# Rolling volatility
fig.add_trace(go.Scatter(x=rolling_std.index, y=rolling_std, 
                         name='Annual Volatility', line=dict(color='red')), 
              row=2, col=1)

# Rolling Sharpe
fig.add_trace(go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe, 
                         name='Sharpe Ratio', line=dict(color='green')), 
              row=3, col=1)

fig.update_layout(height=900, title_text=f"{ticker} Rolling Statistics (252-day window)",
                  showlegend=False)
fig.update_xaxes(title_text="Date", row=3, col=1)
fig.update_yaxes(tickformat='.0%', row=1, col=1)
fig.update_yaxes(tickformat='.0%', row=2, col=1)

fig.show()

### 5.2 For Sectors

In [ ]:
## Rolling Statistics for Sectors

# Calculate rolling statistics for a selected sector
window = 252  # 1 year

# Select sector to analyze (choose the one with most stocks for robust analysis)
selected_sector = 'Technology'  # Or use sector_performance.index[0] for best Sharpe

# Get stocks in this sector
sector_stocks = [t for t in sectors[selected_sector] if t in returns.columns]

# Calculate sector returns (equal-weighted)
sector_daily_returns = returns[sector_stocks].mean(axis=1)

# Rolling calculations
rolling_mean = sector_daily_returns.rolling(window).mean() * 252
rolling_std = sector_daily_returns.rolling(window).std() * np.sqrt(252)
rolling_sharpe = (rolling_mean - RISK_FREE_RATE) / rolling_std

# Create subplots
fig = make_subplots(rows=3, cols=1, 
                    subplot_titles=(f'{selected_sector} - Rolling Annual Return', 
                                   f'{selected_sector} - Rolling Annual Volatility', 
                                   f'{selected_sector} - Rolling Sharpe Ratio'),
                    shared_xaxes=True,
                    vertical_spacing=0.05)

# Rolling return
fig.add_trace(go.Scatter(x=rolling_mean.index, y=rolling_mean, 
                         name='Annual Return', 
                         line=dict(color='blue', width=2)), 
              row=1, col=1)

# Add zero line for reference
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)

# Rolling volatility
fig.add_trace(go.Scatter(x=rolling_std.index, y=rolling_std, 
                         name='Annual Volatility', 
                         line=dict(color='red', width=2)), 
              row=2, col=1)

# Rolling Sharpe with zero line
fig.add_trace(go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe, 
                         name='Sharpe Ratio', 
                         line=dict(color='green', width=2)), 
              row=3, col=1)
fig.add_hline(y=0.5, line_dash="dash", line_color="gray", row=3, col=1)

fig.update_layout(height=900, 
                  title_text=f"{selected_sector} Sector: Rolling Statistics (252-day window, {len(sector_stocks)} stocks)",
                  showlegend=False)
fig.update_xaxes(title_text="Date", row=3, col=1)
fig.update_yaxes(tickformat='.0%', row=1, col=1)
fig.update_yaxes(tickformat='.0%', row=2, col=1)

fig.show()

# Print summary statistics
print(f"Rolling Statistics Summary for {selected_sector} Sector:")
print("-"*60)
print(f"Number of stocks: {len(sector_stocks)}")
print(f"Current annual return: {rolling_mean.iloc[-1]:.2%}")
print(f"Current volatility: {rolling_std.iloc[-1]:.2%}")
print(f"Current Sharpe ratio: {rolling_sharpe.iloc[-1]:.3f}")
print(f"\nHistorical ranges (1-year rolling):")
print(f"  Return: {rolling_mean.min():.1%} to {rolling_mean.max():.1%}")
print(f"  Volatility: {rolling_std.min():.1%} to {rolling_std.max():.1%}")
print(f"  Sharpe: {rolling_sharpe.min():.2f} to {rolling_sharpe.max():.2f}")

## 6. Pair Trading Analysis

In [ ]:
sector_df.Sector.unique()

In [ ]:
sector_df.loc[sector_df['Sector'] == 'Energy']

In [ ]:
# Add this section after the correlation analysis

## Pairs Trading Analysis

# Calculate spread between highly correlated pairs
pairs_to_analyze = [('AAPL', 'MSFT'), ('BAC', 'JPM'), ('CVX', 'COP')]

fig, axes = plt.subplots(len(pairs_to_analyze), 2, figsize=(15, 4*len(pairs_to_analyze)))

for idx, (stock1, stock2) in enumerate(pairs_to_analyze):
    # Calculate normalized prices
    norm1 = prices[stock1] / prices[stock1].iloc[0]
    norm2 = prices[stock2] / prices[stock2].iloc[0]
    
    # Calculate spread
    spread = norm1 - norm2
    spread_mean = spread.mean()
    spread_std = spread.std()
    
    # Plot normalized prices
    ax1 = axes[idx, 0]
    ax1.plot(norm1.index, norm1, label=stock1, linewidth=2)
    ax1.plot(norm2.index, norm2, label=stock2, linewidth=2)
    ax1.set_title(f'{stock1} vs {stock2} - Normalized Prices')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot spread with Bollinger Bands
    ax2 = axes[idx, 1]
    ax2.plot(spread.index, spread, label='Spread', color='black', linewidth=1)
    ax2.axhline(spread_mean, color='blue', linestyle='--', label='Mean')
    ax2.axhline(spread_mean + 2*spread_std, color='red', linestyle='--', label='±2 Std')
    ax2.axhline(spread_mean - 2*spread_std, color='red', linestyle='--')
    ax2.fill_between(spread.index, spread_mean - 2*spread_std, spread_mean + 2*spread_std, alpha=0.1, color='red')
    
    # Mark potential trading signals
    upper_crosses = spread > (spread_mean + 2*spread_std)
    lower_crosses = spread < (spread_mean - 2*spread_std)
    
    ax2.scatter(spread.index[upper_crosses], spread[upper_crosses], color='red', s=20, marker='v', label='Short Signal')
    ax2.scatter(spread.index[lower_crosses], spread[lower_crosses], color='green', s=20, marker='^', label='Long Signal')
    
    ax2.set_title(f'{stock1}-{stock2} Spread (Mean Reversion Opportunities)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Calculate statistics
    correlation = returns[stock1].corr(returns[stock2])
    spread_sharpe = (spread.mean() / spread.std()) * np.sqrt(252)
    
    print(f"\n{stock1}-{stock2} Pair Statistics:")
    print(f"  Correlation: {correlation:.3f}")
    print(f"  Spread Mean: {spread_mean:.3f}")
    print(f"  Spread Std: {spread_std:.3f}")
    print(f"  Spread Sharpe: {spread_sharpe:.3f}")
    print(f"  Current Z-Score: {(spread.iloc[-1] - spread_mean) / spread_std:.2f}")

plt.tight_layout()
plt.show()

## 7. Summary

In [ ]:
## Section 7: Comprehensive Portfolio Analysis Summary

print("="*80)
print("S&P 100 PORTFOLIO ANALYSIS SUMMARY")
print("="*80)

# 1. Universe Overview
print("\n1. UNIVERSE COVERAGE")
print("-"*40)
print(f"Total stocks analyzed: {len(tickers)}")
print(f"Date range: {prices.index[0].strftime('%Y-%m-%d')} to {prices.index[-1].strftime('%Y-%m-%d')}")
print(f"Trading days: {len(prices):,}")
print(f"Sectors covered: {len(sectors)}")


In [ ]:
# 2. Performance Metrics
print("\n2. PERFORMANCE METRICS")
print("-"*40)
print(f"Average annual return: {annual_returns.mean():.2%}")
print(f"Average annual volatility: {annual_volatility.mean():.2%}")
print(f"Average Sharpe ratio: {summary['Sharpe (Static RF)'].mean():.3f}")
print(f"Risk-free rate used: {RISK_FREE_RATE:.2%}")

# Top performers
print(f"\nTop 3 Performers (by Sharpe):")
top_3 = summary.nlargest(3, 'Sharpe (Static RF)')
for ticker in top_3.index:
    print(f"  {ticker}: Return={annual_returns[ticker]:.1%}, Sharpe={top_3.loc[ticker, 'Sharpe (Static RF)']:.2f}")


In [ ]:
# 3. Risk Analysis
print("\n3. RISK CHARACTERISTICS")
print("-"*40)
print(f"Average max drawdown: {summary['Max Drawdown'].mean():.1%}")
print(f"Worst drawdown: {summary['Max Drawdown'].min():.1%} ({summary['Max Drawdown'].idxmin()})")
print(f"Average skewness: {summary['Skewness'].mean():.3f}")
print(f"Stocks with positive skew: {(summary['Skewness'] > 0).sum()}/{len(summary)}")

In [ ]:
# 4. Sector Insights
print("\n4. SECTOR ANALYSIS")
print("-"*40)
print(f"Best sector (Sharpe): {sector_performance.index[0]} ({sector_performance.iloc[0]['Sharpe']:.3f})")
print(f"Highest return sector: {sector_performance['Return'].idxmax()} ({sector_performance['Return'].max():.1%})")
print(f"Lowest risk sector: {sector_performance['Volatility'].idxmin()} ({sector_performance['Volatility'].min():.1%})")
print(f"Average sector correlation: {np.mean(corr_values):.3f}")

In [ ]:
# 5. Portfolio Construction Insights
print("\n5. KEY PORTFOLIO INSIGHTS")
print("-"*40)

# Efficient stocks (high Sharpe, lower correlation)
efficient_stocks = summary[summary['Sharpe (Static RF)'] > 0.7]
print(f"Stocks with Sharpe > 0.7: {len(efficient_stocks)}")

# Diversification opportunities
low_corr_sectors = []
for s1 in sector_correlation.index:
    avg_corr = sector_correlation[s1][sector_correlation.index != s1].mean()
    if avg_corr < 0.6:
        low_corr_sectors.append(s1)

if low_corr_sectors:
    print(f"Diversifying sectors (avg corr < 0.6): {', '.join(low_corr_sectors[:3])}")

In [ ]:
# 6. Actionable Recommendations
print("\n6. PORTFOLIO RECOMMENDATIONS")
print("-"*40)

# Core holdings (high Sharpe, reasonable vol)
core_candidates = summary[(summary['Sharpe (Static RF)'] > 0.6) & 
                          (summary['Annual Volatility'] < 0.3)]
print(f"Core portfolio candidates: {len(core_candidates)} stocks")
print(f"  Suggested allocation: 60-70% in top {min(15, len(core_candidates))} stocks")

# Satellite holdings
growth_stocks = summary[(summary['Annual Return'] > 0.25) & 
                        (summary['Sharpe (Static RF)'] > 0.4)]
print(f"Growth satellites: {len(growth_stocks)} stocks")
print(f"  Suggested allocation: 20-30% for alpha generation")

# Risk management
high_vol = summary[summary['Annual Volatility'] > 0.35]
print(f"High risk stocks (>35% vol): {len(high_vol)} - limit to <5% each")

print("\n" + "="*80)
print("END OF SUMMARY")
print("="*80)

## 8. Save Processed Data

In [ ]:
## Export Data for Portfolio Theory Notebook

import pickle
import os

# Create data directory if it doesn't exist
os.makedirs(os.getcwd() + '/data/processed', exist_ok=True)

# 1. Core price and returns data
prices.to_csv(os.getcwd() + '/data/processed/sp100_prices.csv')
returns.to_csv(os.getcwd() + '/data/processed/sp100_returns.csv')

# 2. Summary statistics (includes Sharpe ratios, volatility, etc.)
summary.to_csv(os.getcwd() + '/data/processed/sp100_summary.csv')

# 3. Sector mappings
pd.Series(sector_mapping).to_csv(os.getcwd() + '/data/processed/sector_mapping.csv')

# 4. Correlation matrix
correlation_matrix = returns.corr()
correlation_matrix.to_csv(os.getcwd() + '/data/processed/correlation_matrix.csv')

# 5. Key parameters as CSV for easy loading
params_df = pd.DataFrame({
    'parameter': ['risk_free_rate', 'start_date', 'end_date', 'n_stocks'],
    'value': [RISK_FREE_RATE, start_date, end_date, len(tickers)]
})
params_df.to_csv(os.getcwd() + '/data/processed/params.csv', index=False)

# 6. Save ticker list
pd.Series(tickers).to_csv(os.getcwd() + '/data/processed/sp100_tickers.csv', index=False)

print("Data Export Complete")
print("-"*50)
print(f"Exported {len(tickers)} stocks")
print(f"Files saved to: {os.getcwd()}/data/processed/")
print("\nFiles created:")
print("  • sp100_prices.csv")
print("  • sp100_returns.csv")
print("  • sp100_summary.csv")
print("  • sector_mapping.csv")
print("  • correlation_matrix.csv")
print("  • sp100_tickers.csv")
print("  • params.csv")